# Principio de Segregación de Interfaces (ISP) — Sistema de citas médicas

## Introducción
El Principio de Segregación de Interfaces (ISP) establece que ningún cliente debería depender de métodos que no usa. Una interfaz "gorda" que agrupa demasiadas responsabilidades obliga a las clases que la implementan a definir métodos irrelevantes para su rol.

## Objetivos
- Mostrar una interfaz gorda `PersonalClinico` que obliga a todos los roles a implementar los mismos cuatro métodos.
- Segregarla en interfaces pequeñas y específicas, de modo que cada clase dependa solo de lo que necesita.

## Ejemplo que viola el ISP

`PersonalClinico` obliga a **cualquier** rol de la clínica a implementar cuatro métodos: atender consultas, realizar cirugías, recetar medicamentos y tomar exámenes de laboratorio.

In [1]:
from abc import ABC, abstractmethod


class PersonalClinico(ABC):
    def __init__(self, nombre: str, cedula_profesional: str) -> None:
        self.nombre = nombre
        self.cedula_profesional = cedula_profesional

    @abstractmethod
    def atender_consulta(self, paciente: str) -> str: ...

    @abstractmethod
    def realizar_cirugia(self, paciente: str) -> str: ...

    @abstractmethod
    def recetar_medicamentos(self, paciente: str) -> str: ...

    @abstractmethod
    def tomar_examenes_laboratorio(self, paciente: str) -> str: ...


class Recepcionista(PersonalClinico):
    def atender_consulta(self, paciente: str) -> str:
        return f"{self.nombre} agenda la cita de {paciente}"

    def realizar_cirugia(self, paciente: str) -> str:
        raise NotImplementedError("Un recepcionista no realiza cirugías")

    def recetar_medicamentos(self, paciente: str) -> str:
        raise NotImplementedError("Un recepcionista no receta medicamentos")

    def tomar_examenes_laboratorio(self, paciente: str) -> str:
        raise NotImplementedError("Un recepcionista no toma exámenes de laboratorio")

In [2]:
recepcionista = Recepcionista(nombre="Marta Ruiz", cedula_profesional="R-001")
print(recepcionista.atender_consulta("Juan"))

try:
    recepcionista.realizar_cirugia("Juan")
except NotImplementedError as error:
    print(f"Falla esperada: {error}")

Marta Ruiz agenda la cita de Juan
Falla esperada: Un recepcionista no realiza cirugías


### Por qué esto es un problema

`Recepcionista` está forzada por el contrato de `PersonalClinico` a declarar `realizar_cirugia`, `recetar_medicamentos` y `tomar_examenes_laboratorio`, tres métodos que nunca podrá ejecutar de verdad. Cualquier código que reciba un `PersonalClinico` genérico y llame a esos métodos puede fallar en tiempo de ejecución con roles como `Recepcionista`. La interfaz agrupa responsabilidades que no todos los roles comparten.

## Versión corregida: interfaces segregadas

Se separan las responsabilidades en cuatro interfaces pequeñas. Cada rol implementa **solo** las que le corresponden.

In [3]:
class AtiendeConsultas(ABC):
    @abstractmethod
    def atender_consulta(self, paciente: str) -> str: ...


class RealizaCirugias(ABC):
    @abstractmethod
    def realizar_cirugia(self, paciente: str) -> str: ...


class PrescribeMedicamentos(ABC):
    @abstractmethod
    def recetar_medicamentos(self, paciente: str) -> str: ...


class RealizaExamenesLaboratorio(ABC):
    @abstractmethod
    def tomar_examenes_laboratorio(self, paciente: str) -> str: ...


class MedicoGeneral(AtiendeConsultas, PrescribeMedicamentos):
    def __init__(self, nombre: str, consultorio: int) -> None:
        self.nombre = nombre
        self.consultorio = consultorio

    def atender_consulta(self, paciente: str) -> str:
        return f"Dr(a). {self.nombre} atiende a {paciente} en el consultorio {self.consultorio}"

    def recetar_medicamentos(self, paciente: str) -> str:
        return f"Dr(a). {self.nombre} receta medicamentos a {paciente}"


class Cirujano(AtiendeConsultas, RealizaCirugias):
    def __init__(self, nombre: str, quirofano: int) -> None:
        self.nombre = nombre
        self.quirofano = quirofano

    def atender_consulta(self, paciente: str) -> str:
        return f"Dr(a). {self.nombre} evalúa a {paciente} antes de la cirugía"

    def realizar_cirugia(self, paciente: str) -> str:
        return f"Dr(a). {self.nombre} realiza cirugía a {paciente} en el quirófano {self.quirofano}"


class RecepcionistaSegregada:
    def __init__(self, nombre: str, turno: str) -> None:
        self.nombre = nombre
        self.turno = turno

    def agendar_cita(self, paciente: str) -> str:
        return f"{self.nombre} agenda la cita de {paciente} (turno {self.turno})"

    def confirmar_asistencia(self, paciente: str) -> str:
        return f"{self.nombre} confirma la asistencia de {paciente}"

In [4]:
medico = MedicoGeneral(nombre="Elena Suárez", consultorio=4)
cirujano = Cirujano(nombre="Ricardo Nieto", quirofano=2)
recepcion = RecepcionistaSegregada(nombre="Marta Ruiz", turno="mañana")

print(medico.atender_consulta("Juan"))
print(medico.recetar_medicamentos("Juan"))
print(cirujano.atender_consulta("Sofía"))
print(cirujano.realizar_cirugia("Sofía"))
print(recepcion.agendar_cita("Juan"))
print(recepcion.confirmar_asistencia("Juan"))

# RecepcionistaSegregada ni siquiera tiene un método realizar_cirugia: no existe la posibilidad
# de que falle al llamarlo porque nunca se le exigió implementarlo.
assert not hasattr(recepcion, "realizar_cirugia")
print("¡Todo correcto!")

Dr(a). Elena Suárez atiende a Juan en el consultorio 4
Dr(a). Elena Suárez receta medicamentos a Juan
Dr(a). Ricardo Nieto evalúa a Sofía antes de la cirugía
Dr(a). Ricardo Nieto realiza cirugía a Sofía en el quirófano 2
Marta Ruiz agenda la cita de Juan (turno mañana)
Marta Ruiz confirma la asistencia de Juan
¡Todo correcto!


### Análisis

- Antes, `Recepcionista` dependía de una interfaz con 4 métodos, de los cuales solo usaba 1.
- Ahora, `RecepcionistaSegregada` ni siquiera hereda de una interfaz clínica: solo tiene los métodos que su rol necesita.
- `MedicoGeneral` implementa `AtiendeConsultas` y `PrescribeMedicamentos`, pero no `RealizaCirugias`; `Cirujano` implementa `AtiendeConsultas` y `RealizaCirugias`, pero no `PrescribeMedicamentos`. Cada clase depende únicamente de lo que usa, y ya no existen métodos con `NotImplementedError`.

## Autoevaluación
- ¿Qué otro rol de la clínica (por ejemplo, un enfermero) podría necesitar una combinación distinta de estas interfaces?

## Referencias
- Martin, R. C. — *The Interface Segregation Principle*.
- [SOLID Principles en Python – Real Python](https://realpython.com/solid-principles-python/)